# Phase 2+3 — Preprocessing + Training

Sets up MONAI data pipelines, builds 3D U-Net, and runs the training loop.

**Kernel:** `gtx-1080-IP`

In [ ]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

## Configuration

Change `CONFIG` to `'full'` for real training.

In [ ]:
import torch
from src.utils.config import load_config
from src.utils.seed import set_seed
from src.utils.logging import init_wandb, finish_wandb, print_log

CONFIG = 'dev'  # 'dev' for fast testing, 'full' for real training
RESUME_FROM = None  # Set to 'checkpoints/last_model.pth' to resume

cfg = load_config(CONFIG)
set_seed(cfg['seed'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Config: {CONFIG}')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

## Initialize WandB (optional)

In [ ]:
run_name = f'brats-3dunet-{CONFIG}'
init_wandb(cfg, run_name=run_name)

## Load Data & Create Splits

In [ ]:
from src.data.dataset import discover_brats_samples, get_monai_file_list
from src.data.splits import create_splits

max_samples = cfg['data'].get('max_samples')
samples = discover_brats_samples(cfg['paths']['data_root'], max_samples=max_samples)

train_samples, val_samples, test_samples = create_splits(
    samples,
    ratios=cfg['data']['split_ratios'],
    seed=cfg['seed'],
    splits_dir=cfg['paths']['splits_dir'],
)

train_files = get_monai_file_list(train_samples)
val_files = get_monai_file_list(val_samples)

print(f'Training samples: {len(train_files)}')
print(f'Validation samples: {len(val_files)}')

## Build Data Pipelines

In [ ]:
from monai.data import CacheDataset, DataLoader, list_data_collate
from src.data.transforms import get_train_transforms, get_val_transforms

train_transforms = get_train_transforms(cfg)
val_transforms = get_val_transforms(cfg)

# Cache rate: 100% for small datasets, 50% for large
cache_rate = 1.0 if max_samples and max_samples <= 10 else 0.5

train_ds = CacheDataset(
    data=train_files,
    transform=train_transforms,
    cache_rate=cache_rate,
    num_workers=cfg['training'].get('num_workers', 2),
)

val_ds = CacheDataset(
    data=val_files,
    transform=val_transforms,
    cache_rate=1.0,
    num_workers=cfg['training'].get('num_workers', 2),
)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg['training']['batch_size'],
    shuffle=True,
    num_workers=cfg['training'].get('num_workers', 2),
    collate_fn=list_data_collate,
    pin_memory=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=1,
    shuffle=False,
    num_workers=cfg['training'].get('num_workers', 2),
    pin_memory=True,
)

print(f'Train batches per epoch: {len(train_loader)}')
print(f'Val batches per epoch: {len(val_loader)}')

## Build Model, Loss, Optimizer, Scheduler

In [ ]:
from src.models.unet3d import get_model
from src.training.losses import get_loss_function

model = get_model(cfg)
loss_fn = get_loss_function(cfg)

opt_cfg = cfg['training']['optimizer']
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=opt_cfg['lr'],
    weight_decay=opt_cfg.get('weight_decay', 1e-5),
)

sch_cfg = cfg['training']['scheduler']
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=sch_cfg['T_0'],
    T_mult=sch_cfg.get('T_mult', 2),
    eta_min=sch_cfg.get('eta_min', 1e-7),
)

## Train!

In [ ]:
from src.training.trainer import Trainer

trainer = Trainer(
    model=model,
    loss_fn=loss_fn,
    optimizer=optimizer,
    scheduler=scheduler,
    train_loader=train_loader,
    val_loader=val_loader,
    cfg=cfg,
    device=device,
)

# Resume from checkpoint if set
if RESUME_FROM:
    trainer.resume_from_checkpoint(RESUME_FROM)

history = trainer.train()

## Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Dice curves
axes[1].plot(history['val_dice_wt'], label='WT')
axes[1].plot(history['val_dice_tc'], label='TC')
axes[1].plot(history['val_dice_et'], label='ET')
axes[1].plot(history['val_dice_mean'], label='Mean', linewidth=2, linestyle='--')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Dice')
axes[1].set_title('Validation Dice')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
finish_wandb()

print('\n✅ Training complete.')
print(f'  Best checkpoint: {cfg["paths"]["checkpoint_dir"]}/best_model.pth')
print(f'  Last checkpoint: {cfg["paths"]["checkpoint_dir"]}/last_model.pth')